In [2]:
%%writefile symtab.l
%{
#include <stdio.h>
#include <string.h>
#include <stdlib.h>
struct symtab {
char name[30];
int type;
} symtab[100];
int sc = 0;
int lookup(char *s) {
int i;
for (i = 0; i < sc; i++)
if (strcmp(symtab[i].name, s) == 0)
return i;
return -1;

}
void insert(char *s) {
if (lookup(s) == -1) {
strcpy(symtab[sc].name, s);
symtab[sc].type = 1;
sc++;
}
}
%}
DIGIT [0-9]
ID [a-zA-Z_][a-zA-Z0-9_]*
%%
"/*"([^*]|\*+[^*/])*\*+"/" { printf("Comment : %s\n", yytext); }
"//".* { printf("Comment : %s\n", yytext); }
{ID} { insert(yytext); printf("Identifier : %s\n", yytext); }
{DIGIT}+ { printf("Constant : %s\n", yytext); }
"+"|"-"|"*"|"/"|"="|"<"|">" { printf("Operator : %s\n", yytext); }
[ \t\n] { /* skip whitespace */ }
. { /* ignore other characters */ }
%%
int yywrap() { return 1; }
int main(int argc, char *argv[]) {
if (argc < 2) {
printf("Usage: %s <input file>\n", argv[0]);
return 1;
}
yyin = fopen(argv[1], "r");
if (!yyin) {
printf("Cannot open file %s\n", argv[1]);
return 1;
}
yylex();
printf("\nSYMBOL TABLE\n");
printf("S.No\tName\n");
int i;
for (i = 0; i < sc; i++)
printf("%d\t%s\n", i + 1, symtab[i].name);
fclose(yyin);
return 0;
}

Writing symtab.l


In [5]:
!apt-get update
!apt-get install -y flex build-essential
!flex symtab.l
!gcc lex.yy.c -lfl -o symtab_lexer

Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:3 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:4 https://cli.github.com/packages stable/main amd64 Packages [356 B]
Get:5 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:6 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [105 kB]
Hit:7 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:8 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [10.7 MB]
Get:9 http://security.ubuntu.com/ubuntu jammy-security/restricted amd64 Packages [7,545 kB]
Get:10 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:11 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Hit:12 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:13 http://security.ubuntu.com/ubuntu jammy-security/universe amd64 Packages [1,314 kB

In [6]:
%%writefile input.txt
int main() {
    /* This is a multi-line comment */
    int x = 10; // This is a single-line comment
    float y = 20.5;
    char z = 'A';
    if (x < y) {
        x = x + 5;
    }
    return 0;
}

Writing input.txt


In [7]:
!./symtab_lexer input.txt

Identifier : int
Identifier : main
Comment : /* This is a multi-line comment */
Identifier : int
Identifier : x
Operator : =
Constant : 10
Comment : // This is a single-line comment
Identifier : float
Identifier : y
Operator : =
Constant : 20
Constant : 5
Identifier : char
Identifier : z
Operator : =
Identifier : A
Identifier : if
Identifier : x
Operator : <
Identifier : y
Identifier : x
Operator : =
Identifier : x
Operator : +
Constant : 5
Identifier : return
Constant : 0

SYMBOL TABLE
S.No	Name
1	int
2	main
3	x
4	float
5	y
6	char
7	z
8	A
9	if
10	return
